In [25]:
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu130

Note: you may need to restart the kernel to use updated packages.


In [26]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [19]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [20]:
import torch
from transformers import AutoModel
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)

model = AutoModel.from_pretrained(
    "genbio-ai/genbio-pathfm",
    trust_remote_code=True
)

model = model.to(device)
model.eval()

Using: cuda


GenBioPathFMModel(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(1, 1536, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (rope_embed): RopePositionEmbedding()
    (blocks): ModuleList(
      (0-39): 40 x SelfAttentionBlock(
        (norm1): LayerNorm((1536,), eps=1e-06, elementwise_affine=True, bias=True)
        (attn): SelfAttention(
          (qkv): Linear(in_features=1536, out_features=4608, bias=False)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1536, out_features=1536, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (norm2): LayerNorm((1536,), eps=1e-06, elementwise_affine=True, bias=True)
        (mlp): SwiGLUFFN(
          (w1): Linear(in_features=1536, out_features=4096, bias=True)
          (w2): Linear(in_features=1536, out_features=4096, bias=True)
          (w3): Linear(in_features=4096, out_features=1

In [21]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.697, 0.575, 0.728),
        std=(0.188, 0.240, 0.187),
    ),
])

In [22]:
from PIL import Image
from glob import glob
from pathlib import Path

image_paths = glob("../data/*/*.png")  # .. first to get out of notebooks folder
if len(image_paths) == 0:
    raise ValueError("No images found in the specified directory: Download at least one data into data/ from hest")
image_path = image_paths[0]  # Select the first image path
print("file name:", Path(image_path).stem)

image = Image.open(image_path).convert("RGB")

x = transform(image).unsqueeze(0).to(device)

print("Input shape:", x.shape)

with torch.inference_mode():
    embedding = model(x)

print("Embedding shape:", embedding.shape)

file name: TENX95_spatial_plots
Input shape: torch.Size([1, 3, 224, 224])
Embedding shape: torch.Size([1, 4608])


In [23]:
print(embedding)

tensor([[ 1.1494,  0.2876,  0.4871,  ..., -0.0458,  0.1730, -0.0860]],
       device='cuda:0')
